# Fine-tuning: Test-subset: BigP300BCI

## Цель эксперимента

Провести финальную честную оценку стратегий обучения на **Test-выборке BigP300BCI**, не использовавшейся для выбора гиперпараметров и стратегий.

Сравниваются:

- scratch
- full fine-tuning (`full_ft`)
- low LR encoder (`low_lr_encoder`)
- partial fine-tuning (`partial_ft`)
- warmup fine-tuning (`warmup`)

---

## Фиксированные условия

Все эксперименты проводятся при одинаковых настройках:

- `lr_encoder = 3e-5`
- `lr_head = 3e-4`
- `weight_decay = 1e-3`
- `warmup_epochs = 3`

Важно:

- гиперпараметры **не подбираются**
- split’ы фиксированы
- нормализация считается только по `Calib_p` конкретного субъекта
- `test_rest` не используется при обучении

---

## Протокол

Для каждого субъекта и каждого уровня калибровки

\[
p \in \{0, 10, 20, 40, 60, 100\}
\]

выполняется:

1. формирование `Calib_p`
2. `train/val` split внутри `Calib_p` (для `p > 0`)
3. обучение модели
4. early stopping по `val_loss`
5. оценка на фиксированном `test_rest`

---

## Метрики

Сохраняются:

- ROC-AUC
- Accuracy
- Binary F1
- Precision
- Recall
- FDR

---

## Архитектура

- Encoder: `UNet1DEncoder`
- Head: Global Average Pooling по времени + Linear (512 → 2)
- Loss: CrossEntropy

---

## Важно про Test

Этот ноутбук используется для **финальной оценки**.  
Результаты из него будут использоваться для итоговых графиков, статистического сравнения и выводов в дипломе.

## 1. Импорты

In [1]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import stage5_utils as u

import model_unet as m

## 2. Конфиги

In [2]:
RUNTIME_CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,
}

In [3]:
PATHS = {
    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoint": Path("/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt"),
    "results_root": Path("/kaggle/working/stage5_results"),
}

## 3. Воспроизводимость

In [4]:
u.set_seed(RUNTIME_CONFIG["seed"])
print("Device:", RUNTIME_CONFIG["device"])

Device: cpu


## 4. Пути

In [5]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": PATHS["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": PATHS["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": PATHS["data_root"] / "bcicompiii-ds2",  
}

In [6]:
GROUPS = ["train", "benchmark", "bcicomp3"]

### Автореестр субъектов

In [7]:
# Автореестр субъектов
def list_subject_ids(group: str):
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        data_dir = DATASETS["bigp3_train"] / "train"
    elif group == "benchmark":
        data_dir = DATASETS["bigp3_benchmark"]/ "benchmark"

    subject_ids = sorted([p.stem for p in data_dir.glob("*.npz")])
    return subject_ids

In [8]:
# Сборка реестра
SUBJECT_REGISTRY = {
    "train": list_subject_ids("train"),
    "benchmark": list_subject_ids("benchmark"),
}

print("Train subjects:", len(SUBJECT_REGISTRY["train"]))
print("Benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("Example train:", SUBJECT_REGISTRY["train"][:5])
print("Example benchmark:", SUBJECT_REGISTRY["benchmark"][:5])


Train subjects: 93
Benchmark subjects: 65
Example train: ['subj_000', 'subj_001', 'subj_002', 'subj_003', 'subj_004']
Example benchmark: ['subj_051', 'subj_052', 'subj_053', 'subj_054', 'subj_055']


## Конфиг для FT-сценариев и scratch на Test

In [9]:
DEV_SUBJECTS = [
    'subj_054', 
    'subj_065', 
    'subj_090', 
    'subj_094'
]

In [10]:
ALL_BENCHMARK_SUBJECTS = SUBJECT_REGISTRY["benchmark"]

FILTERED_BENCHMARK_SUBJECTS = [
    s for s in ALL_BENCHMARK_SUBJECTS
    if s not in DEV_SUBJECTS
]

print("Original benchmark subjects:", len(ALL_BENCHMARK_SUBJECTS))
print("After exclusion:", len(FILTERED_BENCHMARK_SUBJECTS))
print("Excluded:", DEV_SUBJECTS)

Original benchmark subjects: 65
After exclusion: 61
Excluded: ['subj_054', 'subj_065', 'subj_090', 'subj_094']


In [11]:
FINAL_TEST_CONFIG = {
    "tag": "stage5_final_eval_test_benchmark_all_strategies",
    "subjects": FILTERED_BENCHMARK_SUBJECTS,
    "group": "benchmark",

    "p_list": [0, 10, 20, 40, 60, 100],

    # сценарии
    "ft_scenarios": ["full_ft", "low_lr_encoder", "partial_ft", "warmup"],
    "scratch_scenarios": ["scratch"],

    # SCRATCH параметры
    "scratch_lr": 3e-4,
    "scratch_weight_decay": 1e-4,

    # FT параметры
    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
    "warmup_epochs": 3,
}

## Пути для FT и Scratch на Test

In [12]:
RUN_DIR = PATHS["results_root"] / FINAL_TEST_CONFIG["tag"]

ARTIFACT_DIRS = {
    "root": RUN_DIR,
    "ft": RUN_DIR / "ft_strategies",
    "scratch": RUN_DIR / "scratch",
    "history": RUN_DIR / "history",
    "predictions": RUN_DIR / "predictions",
    "tables": RUN_DIR / "tables",
    "figures": RUN_DIR / "figures",
}

for path in ARTIFACT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

RUN_DIR: /kaggle/working/stage5_results/stage5_final_eval_test_benchmark_all_strategies


## Полный прогон FT-стратегий на Test

In [ ]:
final_test_ft_df = u.run_many(
    subject_list=FINAL_TEST_CONFIG["subjects"],
    p_list=FINAL_TEST_CONFIG["p_list"],
    scenario_list=["ssl_ft"],
    group=FINAL_TEST_CONFIG["group"],
    runtime_config=RUNTIME_CONFIG,
    results_root=ARTIFACT_DIRS["ft"],
    encoder_checkpoint=PATHS["encoder_checkpoint"],
    ft_strategy_list=FINAL_TEST_CONFIG["ft_scenarios"],
    lr_encoder=FINAL_TEST_CONFIG["lr_encoder"],
    lr_head=FINAL_TEST_CONFIG["lr_head"],
    weight_decay=FINAL_TEST_CONFIG["weight_decay"],
    warmup_epochs=FINAL_TEST_CONFIG["warmup_epochs"],
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

final_test_ft_summary_path = ARTIFACT_DIRS["ft"] / "summary.csv"
final_test_ft_df.to_csv(final_test_ft_summary_path, index=False)

print("FT test run finished.")
print("Saved to:", final_test_ft_summary_path)
print("Shape:", final_test_ft_df.shape)
display(final_test_ft_df.head())

## Полный прогон Scratch на Test

In [ ]:
# =========================
# Full Test run: scratch on benchmark
# =========================

print("SCRATCH DIR:", ARTIFACT_DIRS["scratch"])
print("Group:", FINAL_TEST_CONFIG["group"])
print("N benchmark subjects:", len(FINAL_TEST_CONFIG["subjects"]))
print("p_list:", FINAL_TEST_CONFIG["p_list"])
print("lr_encoder:", FINAL_TEST_CONFIG["lr_encoder"])
print("lr_head:", FINAL_TEST_CONFIG["lr_head"])
print("weight_decay:", FINAL_TEST_CONFIG["weight_decay"])

final_test_scratch_df = u.run_many(
    subject_list=FINAL_TEST_CONFIG["subjects"],
    p_list=FINAL_TEST_CONFIG["p_list"],
    scenario_list=["scratch"],
    group=FINAL_TEST_CONFIG["group"],
    runtime_config=RUNTIME_CONFIG,
    results_root=ARTIFACT_DIRS["scratch"],
    scratch_lr=FINAL_TEST_CONFIG["scratch_lr"],
    scratch_weight_decay=FINAL_TEST_CONFIG["scratch_weight_decay"],
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

final_test_scratch_summary_path = ARTIFACT_DIRS["scratch"] / "summary.csv"
final_test_scratch_df.to_csv(final_test_scratch_summary_path, index=False)

print("\nScratch test run finished.")
print("Saved summary to:", final_test_scratch_summary_path)
print("Shape:", final_test_scratch_df.shape)

display(final_test_scratch_df.head())